In [ ]:
import torch

from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps, loads

from transformers import AutoTokenizer, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import PeftConfig, PeftModel

import re

In [ ]:
def optimize_circuit_with_slm(
    circuit,
    model,
    tokenizer,
    max_new_tokens=512,
):
    qasm = dumps(circuit)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a quantum circuit compiler. "
                "Rewrite the supplied OpenQASM 3 program into a mathematically "
                "equivalent but simpler OpenQASM 3 program. "
                "The output must implement exactly the same unitary transformation. "
                "Every qubit used in the output must be declared. "
                "Do not introduce measurements, classical bits, resets, or classical "
                "control when the input contains none. "
                "Do not remove qubit declarations. "
                "Return a complete, syntactically valid OpenQASM 3 program and nothing else."
            ),
        },
        {
            "role": "user",
            "content": (
                "Optimize this quantum circuit by reducing gate count and depth "
                "where possible while preserving its exact operation:\n\n"
                f"{qasm}"
            ),
        },
    ]

    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False, 
        )

    # Some tokenizer chat templates don't accept enable_thinking
    except TypeError:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    prompt_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0, prompt_length:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    # Remove Qwen-style thinking
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL,
    ).strip()

    # Remove Markdown code fences
    text = re.sub(
        r"```(?:openqasm|qasm|qasm3)?",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = text.replace("```", "").strip()

    # Sometimes the model may put text before OPENQASM
    openqasm_pos = text.find("OPENQASM")

    if openqasm_pos != -1:
        text = text[openqasm_pos:]

    try:
        optimized_circuit = loads(text)
        return optimized_circuit

    except Exception as e:
        print("Model returned invalid OpenQASM 3.")
        print("Returning original circuit.")
        print("\nGenerated output was:\n")
        print(text)
        print("\nParser error:")
        print(e)

    return circuit

In [ ]:

ADAPTER_PATH = "./qwen35_4b_quantum_optimizer_qlora"

peft_config = PeftConfig.from_pretrained(
    ADAPTER_PATH
)

BASE_MODEL = peft_config.base_model_name_or_path

print("Adapter was trained from:")
print(BASE_MODEL)

print("\nLoRA target modules:")
print(peft_config.target_modules)

print("\nLoRA rank:")
print(peft_config.r)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForMultimodalLM.from_pretrained(
    BASE_MODEL,

    quantization_config=quantization_config,

    device_map={"": 0},

    torch_dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False,
)

model.eval()
model.config.use_cache = True

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

if hasattr(model, "base_model"):
    if hasattr(model.base_model, "gradient_checkpointing_disable"):
        model.base_model.gradient_checkpointing_disable()

torch.cuda.empty_cache()

print("\nLoaded successfully.")
print("Base model:", BASE_MODEL)
print("Training:", model.training)
print("use_cache:", model.config.use_cache)
print(
    "gradient checkpointing:",
    getattr(model, "is_gradient_checkpointing", "unknown"),
)

In [ ]:
def optimize_circuit(qc):
    print("Old Circuit:")
    print(qc)

    optimized_qc = optimize_circuit_with_slm(
        qc,
        model,
        tokenizer,
    )

    print("SLM Optimized Circuit:")
    print(optimized_qc)

In [ ]:
qc = QuantumCircuit(1)

qc.h(0)
qc.h(0)

optimize_circuit(qc)

In [ ]:
model.eval()
model.config.use_cache = True

qc = QuantumCircuit(2)

qc.cx(0, 1)
qc.x(0)
qc.x(0)
qc.cx(0, 1)

optimize_circuit(qc)

In [ ]:
qc = QuantumCircuit(1)

qc.s(0)
qc.sdg(0)

optimize_circuit(qc)

In [ ]:
qc = QuantumCircuit(2)

qc.x(0)
qc.x(0)

qc.h(1)
qc.h(1)

qc.cx(0, 1)
qc.cx(0, 1)

qc.z(0)
qc.z(0)

optimize_circuit(qc)

In [ ]:
qc = QuantumCircuit(1)

qc.h(0)
qc.x(0)
qc.x(0)

optimize_circuit(qc)

In [ ]:
qc = QuantumCircuit(2)

qc.h(0)

qc.cx(0, 1)
qc.z(1)
qc.z(1)
qc.cx(0, 1)

qc.x(0)
qc.x(0)

qc.h(1)

optimize_circuit(qc)

In [ ]:
from qiskit import QuantumCircuit
import numpy as np

qc = QuantumCircuit(3)

qc.h(0)

qc.cx(0, 1)

qc.x(2)
qc.x(2)

qc.cx(1, 2)

qc.rz(np.pi / 5, 0)
qc.rz(-np.pi / 5, 0)

qc.h(1)
qc.h(1)

qc.cx(1, 2)

qc.s(2)
qc.sdg(2)

qc.cx(0, 1)

qc.h(2)

optimize_circuit(qc)

In [ ]:
qc = QuantumCircuit(3)

qc.h(0)

qc.x(2)
qc.x(2)

qc.cp(np.pi / 2, 1, 0)

qc.h(1)
qc.h(1)

qc.cp(np.pi / 4, 2, 0)

qc.h(1)

qc.cx(0, 2)
qc.cx(0, 2)

qc.cp(np.pi / 2, 2, 1)

qc.rz(np.pi / 7, 0)
qc.rz(-np.pi / 7, 0)

qc.h(2)

qc.swap(0, 2)

optimize_circuit(qc)

In [ ]:
qc = QuantumCircuit(3)

qc.x(2)
qc.x(2)

qc.h(0)

qc.rz(np.pi / 6, 1)
qc.rz(-np.pi / 6, 1)

qc.cx(0, 1)

qc.h(2)
qc.h(2)

qc.s(0)
qc.sdg(0)

qc.cx(1, 2)

qc.rx(np.pi / 5, 2)
qc.rx(-np.pi / 5, 2)

qc.t(1)
qc.tdg(1)

optimize_circuit(qc)

In [ ]:
qc = QuantumCircuit(4)

qc.x(3)
qc.x(3)

qc.rz(np.pi / 6, 2)
qc.rz(-np.pi / 6, 2)

qc.h(0)

qc.z(2)
qc.z(2)

qc.h(1)

qc.cx(0, 2)

qc.s(3)
qc.sdg(3)

qc.cx(1, 2)

qc.cx(0, 3)

qc.h(2)
qc.h(2)

qc.cx(1, 3)

qc.rx(np.pi / 5, 2)
qc.rx(-np.pi / 5, 2)

qc.t(3)
qc.tdg(3)

qc.h(0)

qc.cz(2, 3)
qc.cz(2, 3)

qc.h(1)

optimize_circuit(qc)